# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Jadzia Afia Ohenewaa Mantey]
**Student ID:** [23672028]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [25]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [26]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response


# Test the function
response = ask_llm("What is microfinance?")

print("Answer:")
print(response.choices[0].message.content)

print("\nToken usage:")
print(response.usage)



Answer:
Microfinance refers to the provision of small-scale financial services, typically to low-income individuals or households who lack access to traditional banking services. The goal of microfinance is to provide financial inclusion and empowerment to those who are often excluded from the formal financial system.

Microfinance services can include:

1. **Microloans**: Small loans, often with flexible repayment terms, to help individuals start or expand a business, pay for education or healthcare, or cover unexpected expenses.
2. **Savings accounts**: Opportunities for individuals to save money in a secure and accessible way, often with interest earned on deposits.
3. **Insurance**: Affordable insurance products, such as life insurance, health insurance, or crop insurance, to help protect against unexpected events.
4. **Money transfer services**: Convenient and affordable ways to send and receive money, often across long distances.
5. **Financial education**: Training and support t

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** The system role gives the model high-level instructions about how it should behave throughout the conversation.Like:'You are a careful financial analyst.Do not invent information. The user role contains the actual request or information that the model needs to respond to. Like:Summarize this loan application in three sentences.

2.A token is a small piece of text that an LLM processes. It can be a short word or part of a longer word.API providers bill per token because different requests require very different different amounts of tokens for computation.


### Part 1.2 — Temperature: the randomness dial

In [27]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
question = "Suggest a name for a savings product for market traders in Accra."

# Temperature = 0.0
print("===== TEMPERATURE 0.0 =====")

for i in range(5):
    response = ask_llm(
        question,
        temperature=0.0
    )
    print(f"{i+1}. {response.choices[0].message.content}")


# Temperature = 1.2
print("\n===== TEMPERATURE 1.2 =====")

for i in range(5):
    response = ask_llm(
        question,
        temperature=1.2
    )
    print(f"{i+1}. {response.choices[0].message.content}")

===== TEMPERATURE 0.0 =====
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Saver**: "Sika" is the Ghanaian word for money, so this name incorporates a local touch.
4. **Market Mobi**: This name suggests mobility and flexibility, which could appeal to market traders who need easy access to their savings.
5. **Kokroko Savings**: "Kokroko" is a Ghanaian term for a collective savings scheme, so this name could evoke a sense of community and shared savings goals.
6. **Accra Advantage**: This name highlights the benefits of saving with a product specifically designed for market traders in Accra.
7. **Suzyo Savings**: "Suzyo" is a Ghanaian word for "save" or "keep", so this name incorporates a local language and emphasizes the importance of saving.



**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0.0, the answers were generally much more consistent, with the model tending to give the same or very similar suggestions each time.

At temperature 1.2, the answers were more varied and creative, produciing different suggestions across the five runs.

For the loan decision-support system, a low temperature (around 0.0–0.2) is more appropriate. The system needs to be consistent and factual when summarizing applications and extracting information. A higher temperature could introduce unnecessary variation or increase the risk of the model making unsupported assumptions.



---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [28]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [29]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

# =========================
# SUMMARY PROMPT V1
# =========================

SUMMARY_PROMPT_V1 = "Summarize this:"

print("===== V1: L002 =====")
response_v1_l002 = ask_llm(
    SUMMARY_PROMPT_V1 + "\n\n" + LETTERS["L002"],
    temperature=0.7
)
v1_l002 = response_v1_l002.choices[0].message.content
print(v1_l002)

print("\n===== V1: L006 =====")
response_v1_l006 = ask_llm(
    SUMMARY_PROMPT_V1 + "\n\n" + LETTERS["L006"],
    temperature=0.7
)
v1_l006 = response_v1_l006.choices[0].message.content
print(v1_l006)


# =========================
# SUMMARY PROMPT V2
# =========================

SUMMARY_SYSTEM_V2 = """
You are an assistant to a microfinance loan officer.

Summarize loan applications accurately and neutrally.
Use only facts explicitly stated in the application.
Do not invent, assume, or infer missing information.
Keep the summary to 3-4 sentences.
Do not make a final lending decision.
"""

SUMMARY_PROMPT_V2 = """
Summarize this loan application:

{letter_text}
"""


print("\n===== V2: L002 =====")
response_v2_l002 = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0
)
v2_l002 = response_v2_l002.choices[0].message.content
print(v2_l002)

print("\n===== V2: L006 =====")
response_v2_l006 = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0
)
v2_l006 = response_v2_l006.choices[0].message.content
print(v2_l006)


# =========================
# SIDE-BY-SIDE COMPARISON
# =========================

print("\n" + "=" * 70)
print("L002 — V1 vs V2")
print("=" * 70)

print("\n--- V1 ---")
print(v1_l002)

print("\n--- V2 ---")
print(v2_l002)


print("\n" + "=" * 70)
print("L006 — V1 vs V2")
print("=" * 70)

print("\n--- V1 ---")
print(v1_l006)

print("\n--- V2 ---")
print(v2_l006)

===== V1: L002 =====
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season. He has no collateral to offer but promises to repay the loan as soon as possible.

===== V1: L006 =====
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He claims to be business-minded and promises to repay the loan within a year when his businesses are successful. However, he has no collateral to offer and is relying on his personal trustworthiness as assurance.

===== V2: L002 =====
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but expects it to improve af

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** V1 problems: It was vague and could produce overly general or unsupported summaries. V2 fixes this by requiring factual, neutral summaries. For example, L006 says Kofi “has not started any of these yet,” so the summary should not imply he already runs those businesses.
“No invented details” is essential because false information could affect a real loan decision. This failure is called hallucination—when an LLM generates unsupported or false information.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [30]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

In [31]:
import json
import pandas as pd

EXTRACT_PROMPT = """
Extract information from the loan application and return ONLY a valid JSON object.

The JSON must contain EXACTLY these keys:
{{
  "applicant_name": "string",
  "amount_ghs": "number",
  "purpose": "string",
  "monthly_profit_ghs": "number or null",
  "has_collateral_or_guarantor": "boolean or null",
  "repayment_months": "number or null"
}}

Rules:
- Use only information explicitly stated in the letter.
- If a field is not stated, use null.
- Do not guess or infer missing information.
- Return ONLY the JSON object.
- Do not include explanations, comments, or Markdown.

Worked example:

Letter:
I am Ama Mensah. I need GHS 6,000 to buy a refrigerator.
My shop makes GHS 1,200 profit each month. My brother will
guarantee the loan. I will repay it over 12 months.

Correct JSON:
{{
  "applicant_name": "Ama Mensah",
  "amount_ghs": 6000,
  "purpose": "buy a refrigerator",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 12
}}

Now extract the information from this letter:

{letter_text}
"""


def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)

    try:
        response = ask_llm(
            prompt,
            temperature=0.0
        )

        result = response.choices[0].message.content.strip()

        # Remove Markdown JSON fences if the model adds them
        if result.startswith("```json"):
            result = result[len("```json"):].strip()

        if result.startswith("```"):
            result = result[len("```"):].strip()

        if result.endswith("```"):
            result = result[:-3].strip()

        data = json.loads(result)

        return data

    except (json.JSONDecodeError, AttributeError, KeyError, TypeError) as e:
        print(f"WARNING: Could not parse response: {e}")
        return None

    except Exception as e:
        print(f"WARNING: LLM request failed: {e}")
        return None


results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
    else:
        extracted = {"letter_id": letter_id}

    results.append(extracted)


df = pd.DataFrame(results)

columns = ["letter_id"] + [
    col for col in df.columns if col != "letter_id"
]

df = df[columns]

display(df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers for my poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** Few-shot example: It should not come from the six test lettters because that could leak the expected answers into the prompt and make the evaluation unfair.
“Use null, do not guess: Without it, the model may invent or infer missing information. Using null forces it to admit when information isnt provided.
Temperature = 0: Extraction needs consistent predictable results. Creative tasks benefit from higher temperature because more variation and originality are desirable.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [20]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.


BRIEF_PROMPT = """
You are assisting a microfinance loan officer with decision support.

Review the loan application and the extracted information below.

Produce a brief with EXACTLY these four sections:

1. Strengths
- Give bullet points grounded only in the application.

2. Risks / red flags
- Give bullet points based only on information in the application.
- Do not invent risks or facts.

3. Missing information the officer should request
- List important information that is not provided and would help the officer assess the application.

4. Suggested next step
- Suggest an appropriate process step, such as:
  "invite for interview"
  "request documents"
  "verify information"
  "flag for senior review"
- Do NOT recommend "approve" or "reject".

IMPORTANT:
- The final lending decision must always be made by a human loan officer.
- Do not invent or assume facts.
- Distinguish clearly between stated facts and missing information.
- Be factual, neutral, and concise.

LOAN APPLICATION:
{letter_text}

EXTRACTED INFORMATION:
{extracted_json}
"""


def generate_brief(letter_text, extracted_data):
    prompt = BRIEF_PROMPT.format(
        letter_text=letter_text,
        extracted_json=json.dumps(extracted_data, indent=2)
    )

    response = ask_llm(
        prompt,
        temperature=0.0,
        max_tokens=500
    )

    return response.choices[0].message.content


# Generate briefs for all six applications
briefs = {}

for letter_id, letter_text in LETTERS.items():
    row = df[df["letter_id"] == letter_id]

    if len(row) > 0:
        extracted_data = row.iloc[0].to_dict()
    else:
        extracted_data = {}

    briefs[letter_id] = generate_brief(
        letter_text,
        extracted_data
    )


# Print the three requested applications
for letter_id in ["L001", "L002", "L006"]:
    print("=" * 70)
    print(f"BRIEF FOR {letter_id}")
    print("=" * 70)
    print(briefs[letter_id])
    print()


BRIEF FOR L001
## Step 1: Identify the strengths of the loan application
The applicant, Akosua Mensah, has been selling provisions at Makola Market for 12 years, indicating a stable business history. She has a current monthly profit of GHS 900 and has saved GHS 2,500 through the susu scheme over two years without missing a contribution, showing a good savings record and financial discipline. Additionally, she has a guarantor, her sister, who is a teacher, which could provide an added layer of security for the loan.

## Step 2: Determine the risks or red flags in the application
The applicant is requesting a loan of GHS 8,000, which is a significant amount compared to her monthly profit and savings. The repayment plan of GHS 450 over 20 months may be feasible based on her stated income, but there is no detailed information on her expenses or other financial obligations that could affect her ability to repay the loan.

## Step 3: List the missing information that would be helpful for the

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** L003 vs L006: Yes. L003 should identify strengths such as an established registered business, three apprentices, consistent profit, sales records, and a fixed deposit that can be pledged. L006 should flag that the businesses have not started, there is no collateral, and there is no proven business income. These are appropriate differences.

2.Practical: The LLM may miss important financial information or make errors, so a human needs to verify the application.
Ethical: Loan decisions can significantly affect people's lives. An AI should not make the final decision without human oversight, especially when its reasoning may be incomplete or biased.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** f597073

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [21]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

# Compare extracted values against GOLD values

import pandas as pd

gold_ids = ["L001", "L003", "L006"]

fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]


def normalize_text(value):
    if value is None:
        return None
    return str(value).strip().lower()


def values_match(field, predicted, expected):
    # Names and purpose: case-insensitive comparison
    if field in ["applicant_name", "purpose"]:
        return normalize_text(predicted) == normalize_text(expected)

    # Numbers, booleans, and null: exact comparison
    return predicted == expected


rows = []

for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in gold_ids:
        predicted = df.loc[
            df["letter_id"] == letter_id, field
        ].iloc[0]

        expected = GOLD[letter_id][field]

        correct = values_match(field, predicted, expected)

        row[letter_id] = "✓" if correct else "✗"

        if correct:
            correct_count += 1

    row["accuracy"] = f"{correct_count}/3 ({correct_count / 3:.1%})"
    rows.append(row)


accuracy_table = pd.DataFrame(rows)

display(accuracy_table)

,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,3/3 (100.0%)
1,amount_ghs,✓,✓,✓,3/3 (100.0%)
2,purpose,✗,✗,✗,0/3 (0.0%)
3,monthly_profit_ghs,✓,✓,✗,2/3 (66.7%)
4,has_collateral_or_guarantor,✓,✓,✓,3/3 (100.0%)
5,repayment_months,✓,✓,✓,3/3 (100.0%)


### Part 4.2 — Reliability: is the system consistent?

In [22]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

import json

def extract_fields(letter_text, temperature=0.0):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)

    try:
        response = ask_llm(
            prompt,
            temperature=temperature
        )

        result = response.choices[0].message.content.strip()

        # Remove Markdown JSON fences
        if result.startswith("```json"):
            result = result[len("```json"):].strip()
        elif result.startswith("```"):
            result = result[len("```"):].strip()

        if result.endswith("```"):
            result = result[:-3].strip()

        return json.loads(result)

    except Exception as e:
        print(f"WARNING: Could not parse response: {e}")
        return None

In [23]:
# Test L004 five times at each temperature

results_by_temp = {}

for temperature in [0.0, 1.0]:
    results = []

    for i in range(5):
        result = extract_fields(
            LETTERS["L004"],
            temperature=temperature
        )
        results.append(result)

    results_by_temp[temperature] = results

    # Keep only valid JSON results
    valid_results = [r for r in results if r is not None]

    # Convert results to comparable strings
    unique_results = set(
        json.dumps(r, sort_keys=True)
        for r in valid_results
    )

    print(f"\nTemperature = {temperature}")
    print(f"Valid JSON: {len(valid_results)}/5")
    print(f"Unique outputs: {len(unique_results)}")
    print(f"Identical across all valid runs: {len(unique_results) == 1}")

    for i, result in enumerate(results, 1):
        print(f"\nRun {i}:")
        print(result)


Temperature = 0.0
Valid JSON: 5/5
Unique outputs: 3
Identical across all valid runs: False

Run 1:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm at Nsawam', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}

Run 2:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}

Run 3:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}

Run 4:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}

Run 5:
{'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose'

### Part 4.3 — Hallucination probing

In [24]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

# ==============================
# ADVERSARIAL TEST 1
# ==============================

adversarial_question = """
What is the applicant's credit score?
"""

response_1 = ask_llm(
    adversarial_question + "\n\n" + LETTERS["L002"],
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0
)

output_1 = response_1.choices[0].message.content

print("TEST 1 — Missing information")
print("-" * 50)
print(output_1)


# ==============================
# ADVERSARIAL TEST 2
# ==============================

irrelevant_text = """
The weather forecast for Accra is sunny today with temperatures
around 30 degrees Celsius. There may be some rain later in the week.
"""

output_2 = extract_fields(irrelevant_text)

print("\nTEST 2 — Irrelevant input")
print("-" * 50)
print(json.dumps(output_2, indent=2))

TEST 1 — Missing information
--------------------------------------------------
The applicant, Kwame Boateng, is requesting a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He is a commercial driver in Kumasi and expects his business to improve after the festive season. The applicant does not have collateral to offer at the moment. There is no mention of the applicant's credit score in the loan application.

TEST 2 — Irrelevant input
--------------------------------------------------
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** Extraction accuracy: The model achieved the accuracy shown in my evaluation table. The purpose field was hardest because applicants describe purposes in different ways, making exact matching difficult.
Reliability: Temperature 0.0 produced more consistent results, while 1.0 produced more variation. For a production loan system, low temperature is preferable for reliable extraction.
Hallucination: The adversarial tests showed whether the model invented missing information. The risk can be reduced by explicitly saying “use null, do not guess,” using low temperature, validating JSON programmatically, and keeping a human loan officer in the loop.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**Unfair harm: Applicants with poor English or less formal writing could be judged more negatively even if their businesses are strong. This could unfairly disadvantage less-educated applicants or people who communicate better in local languages.
Third-party API: Sending loan letters to an overseas provider creates privacy and data-protection risks. Before deployment in Ghana, I would check Ghana's data-protection requirements, the provider's data-retention and security policies, where data is stored/processed, whether it is used for model training, and whether appropriate consent and data-processing agreements are in place.

Two safeguards:
Mandatory human review: The LLM can recommend next steps, but a qualified loan officer must make and document the final decision.

Monitoring and appeals: Log model outputs and decisions, regularly test for bias and errors, and provide applicants with a way to challenge or appeal decisions.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** Prompting as engineering: Prompt iteration is similar to hyperparameter tuning because both involve testing changes and evaluating which setup gives better results. The difference is that prompts change the instructions given to an already-trained model, while hyperparameters in Lab 3 changed how the model was trained or learned.

Trust: I would not trust the system to run unattended because loan decisions are high-stakes. The most important result was the adversarial/hallucination test, because inventing information that is not in an application could directly lead to an unfair loan decision.

Cost and scale: Multiply the average total_tokens from your response.usage results by 1,000 applications. For example, if one application uses 1,000 tokens, 1,000 applications would require about 1,000,000 tokens per month. This makes provider pricing, free-tier limits, rate limits, and reliability important when choosing an API.

Looking back: Calling an API is usually better here because the foundation model is already trained, so we can build the system quickly without needing huge datasets, computing resources, or training expertise. Training our own model could make sense when we have large amounts of specialized data, strict privacy requirements, or a need for highly customized behavior that an API cannot provide.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.